# IMPORTS

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

# SPLIT DATA

**The class ids are numbered 1-4 where 1 represents World, 2 represents Sports, 3 represents Business and 4 represents Sci/Tech.**

**Load dữ liệu**

In [16]:
# ── 1. Load dữ liệu
df_train = pd.read_csv('/content/train.csv', encoding='utf-8')
df_test  = pd.read_csv('/content/test.csv', encoding='utf-8')

In [17]:
cols = ['label', 'title', 'description']

df_train.columns = cols
df_test.columns = cols
print('Train columns:', df_train.columns.tolist())
print('Test  columns:', df_test.columns.tolist())

Train columns: ['label', 'title', 'description']
Test  columns: ['label', 'title', 'description']


**Gộp cột title và description**

In [18]:
df_train['text'] = df_train['title'] + ' ' + df_train['description']
df_train = df_train[['label', 'text']]

df_test['text'] = df_test['title'] + ' ' + df_test['description']
df_test = df_test[['label', 'text']]

**Kiểm tra leak**

In [19]:
# ── Kiểm tra Data Leakage giữa df_train và df_test
train_texts = set(df_train['text'].unique())
test_texts = set(df_test['text'].unique())

# Tìm giao điểm (các text xuất hiện ở cả 2 tập)
leakage_texts = train_texts.intersection(test_texts)

print(f"--- KIỂM TRA DATA LEAKAGE ---")
print(f"Số lượng text duy nhất trong Train: {len(train_texts):,}")
print(f"Số lượng text duy nhất trong Test : {len(test_texts):,}")

if len(leakage_texts) > 0:
    print(f"\n⚠️ CẢNH BÁO: Phát hiện {len(leakage_texts)} văn bản bị trùng lặp (leak) giữa tập Train và Test!")
    print(f"Tỷ lệ leak so với tập Test: {len(leakage_texts)/len(test_texts)*100:.2f}%")

    # Hiển thị thử 5 mẫu bị leak
    print("\nVí dụ 5 mẫu bị leak:")
    for i, txt in enumerate(list(leakage_texts)[:5]):
        print(f"{i+1}. {txt[:100]}...")
else:
    print("\n✅ Tuyệt vời: Không phát hiện data leak (trùng lặp nội dung) giữa Train và Test.")
print("----------------------------")

--- KIỂM TRA DATA LEAKAGE ---
Số lượng text duy nhất trong Train: 120,000
Số lượng text duy nhất trong Test : 7,600

✅ Tuyệt vời: Không phát hiện data leak (trùng lặp nội dung) giữa Train và Test.
----------------------------


**Gộp data**

In [20]:
# ── 2. Gộp 2 tập lại
df_all = pd.concat([df_train, df_test], ignore_index=True)
print(f'\nTổng sau khi gộp: {len(df_all):,} mẫu')
print(df_all['label'].value_counts())


Tổng sau khi gộp: 127,600 mẫu
label
3    31900
4    31900
2    31900
1    31900
Name: count, dtype: int64


**Kiểm tra trùng lặp và loại bỏ trùng lặp**

In [21]:
# Kiểm tra trùng lặp và thiếu nhất quán nhãn giữa 'text' và 'label'
print("--- Kiểm tra trùng lặp nội dung 'text' ---")
duplicated_text_mask = df_all.duplicated(subset=['text'], keep=False)
duplicated_texts_df = df_all[duplicated_text_mask].sort_values(by='text')

if duplicated_texts_df.empty:
    print("Không tìm thấy nội dung 'text' trùng lặp.")
else:
    print(f"Tìm thấy {len(duplicated_texts_df)} hàng có nội dung 'text' trùng lặp. Tổng số 'text' duy nhất bị trùng: {duplicated_texts_df['text'].nunique()}")

    print("\n--- Kiểm tra thiếu nhất quán nhãn 'label' cho các 'text' trùng lặp ---")
    inconsistent_labels = []
    for text, group in duplicated_texts_df.groupby('text'):
        if group['label'].nunique() > 1:
            inconsistent_labels.append(group)

    if not inconsistent_labels:
        print("Không tìm thấy sự thiếu nhất quán trong nhãn 'label' đối với các nội dung 'text' trùng lặp.")
    else:
        print(f"Tìm thấy {len(inconsistent_labels)} nội dung 'text' có nhãn 'label' không nhất quán:")
        for group_df in inconsistent_labels:
            print(f"\nNội dung Text: '{group_df['text'].iloc[0]}'\nCác nhãn label tương ứng:\n{group_df[['text', 'label']]}")

print("--------------------------------------------------")

--- Kiểm tra trùng lặp nội dung 'text' ---
Không tìm thấy nội dung 'text' trùng lặp.
--------------------------------------------------


**Tách train/val/test**

In [22]:
# ── 3. Tách X, y
X = df_all['text']
y = df_all['label']

# ── 4. Chia train / val / test (70 / 10 / 20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,
    random_state=42,
    stratify=y_temp
)

# ── 5. Kiểm tra kết quả
print(f'\n✅ Train : {len(X_train):,} mẫu')
print(f'✅ Val   : {len(X_val):,} mẫu')
print(f'✅ Test  : {len(X_test):,} mẫu')

print('\n--- Phân phối nhãn Train ---')
print(y_train.value_counts(normalize=True).round(3))

print('\n--- Phân phối nhãn Val ---')
print(y_val.value_counts(normalize=True).round(3))

print('\n--- Phân phối nhãn Test ---')
print(y_test.value_counts(normalize=True).round(3))


✅ Train : 89,320 mẫu
✅ Val   : 12,760 mẫu
✅ Test  : 25,520 mẫu

--- Phân phối nhãn Train ---
label
4    0.25
3    0.25
1    0.25
2    0.25
Name: proportion, dtype: float64

--- Phân phối nhãn Val ---
label
2    0.25
4    0.25
3    0.25
1    0.25
Name: proportion, dtype: float64

--- Phân phối nhãn Test ---
label
1    0.25
3    0.25
2    0.25
4    0.25
Name: proportion, dtype: float64


In [23]:
# ── 6. Lưu lại thành file CSV
pd.DataFrame({'text': X_train, 'label': y_train}).to_csv('train_split.csv', index=False)
pd.DataFrame({'text': X_val,   'label': y_val}).to_csv('val_split.csv',   index=False)
pd.DataFrame({'text': X_test,  'label': y_test}).to_csv('test_split.csv',  index=False)
print('\n💾 Đã lưu: train_split.csv | val_split.csv | test_split.csv')


💾 Đã lưu: train_split.csv | val_split.csv | test_split.csv
